In [ ]:
import os
import glob
import scipy
import numpy as np
import pandas as pd
import scanpy as sc
import pybiomart
import anndata
import seaborn as sns
import matplotlib.pyplot as plt
import decoupler

from tqdm.auto import tqdm
from scipy.spatial.distance import euclidean

from utils_correlations import load_data_mouse, get_gene_conversion_dict, replace_greek, compute_weighted_mean, pool_adata
from utils_name_maps import cytokine_name_map

In [ ]:
def invert_log1p(adata):
    # y = log(x+1)
    # x = exp(y) - 1
    adata.X = scipy.sparse.csr_matrix(np.exp(np.array(adata.X.todense())) - 1)
    return adata

def map_gene_names(adata, conversion_dict):
    adata.var.loc[:, "gene_name"] = adata.var.index.map(conversion_dict)
    adata = adata[:, ~adata.var.gene_name.isna()]
    adata.var.set_index("gene_name", inplace=True)
    return adata

def load_mouse_data(
    cytokine_name_map: dict,
    from_scratch = False,
    save: bool = True,
):
    """
    Files were downloaded separately per cell type from: https://www.immune-dictionary.org/app/home
    Conversion from .rds to .h5ad was done with the script below, directly in R.
    input_dir <- "/Users/soeren.becker/repos/celltype_rds_files"
    output_dir <- "/Users/soeren.becker/repos/celltype_rds_files"
    
    # Create output folder if it doesn't exist
    if(!dir.exists(output_dir)) dir.create(output_dir)
    
    # List all .rds files in the folder
    rds_files <- list.files(input_dir, pattern = "\\.rds$", full.names = TRUE)
    
    # Loop over each file
    for(file_path in rds_files){
      # Get base name without extension
      file_name <- tools::file_path_sans_ext(basename(file_path))
      message("Processing: ", file_name)
    
      # Load Seurat object
      seurat_obj <- readRDS(file_path)
    
      # Convert to SingleCellExperiment
      sce <- as.SingleCellExperiment(seurat_obj)
    
      # Save as h5ad
      output_file <- file.path(output_dir, paste0(file_name, ".h5ad"))
      writeH5AD(sce, output_file)
    
      message("Saved: ", output_file)
    }
    """
    path = "/Users/soeren.becker/repos/HumanCytokineDict/data/mouse_cytokine_dict/mouse_cytokine_dict_all.h5ad"
    if from_scratch or not os.path.exists(path):
        paths = glob.glob("/Users/soeren.becker/repos/HumanCytokineDict/data/mouse_cytokine_dict/celltype_rds_files/*.h5ad")
        all_adatas = []
        for path in tqdm(paths):
            all_adatas.append(sc.read_h5ad(path))
        adata = anndata.concat(all_adatas, axis=0)
        adata.obs.rename({"sample": "cytokine"}, axis=1, inplace=True)
        adata.obs.loc[:, "cytokine"] = adata.obs.loc[:, "cytokine"].map(lambda x: replace_greek(cytokine_name_map.get(x,x)))
        if save:
            save_path = "/Users/soeren.becker/repos/HumanCytokineDict/data/mouse_cytokine_dict/mouse_cytokine_dict_all.h5ad"
            print(f"Saving {save_path}")
            adata.write_h5ad(save_path)
        return adata
    else:
        return sc.read_h5ad(path)

In [ ]:
adata = load_mouse_data(cytokine_name_map, from_scratch=True, save=False)

In [ ]:
adata.shape # this is different from what is reported in the paper as total number of cells: 386,703.

In [ ]:
adata.obs.head()

In [ ]:
adata.obs.celltype.value_counts()

In [ ]:
adata.layers["raw"] = adata.X.copy()

### number of cells per condition

In [ ]:
df_pivot = adata.obs.groupby("celltype").cytokine.value_counts()\
    .to_frame()\
    .reset_index()\
    .pivot(
        index="cytokine",
        columns="celltype",
        values="count",
    )

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(4.5, 9))
sns.heatmap(
    data=df_pivot,
    cmap="Reds",
    vmin=0,
    vmax=100,
    annot=True,
    ax=ax,
    square=False,
    cbar=False,
    annot_kws={"fontsize": 6, "family": "Arial"},
    fmt=""
)
plt.xlabel("")
plt.ylabel("")
plt.xticks(0.5+np.arange(df_pivot.shape[1]), df_pivot.columns, fontsize=6, family="Arial")
plt.yticks(0.5+np.arange(df_pivot.shape[0]), df_pivot.index, fontsize=6, family="Arial")
plt.show()

In [ ]:
adata.X.min(), adata.X.max(), adata.X.mean()

In [ ]:
sums_cells = np.asarray(adata.X.sum(axis=1)).squeeze()
print(f"Min UMI per cell: {sums_cells.min()}") # matches the preprocessing description in Cui et al. ("scRNA-seq data pre-processing")
sums_expressed_genes = np.asarray(np.sum(adata.X > 0, axis=1)).squeeze()
print(f"Min # expressed genes per cell: {sums_expressed_genes.min()}")

In [ ]:
sns.histplot(sums_cells)
plt.title("Distribution of transcript counts per cell")
plt.show()

In [ ]:
sc.pp.normalize_total(adata, target_sum=10_000)

In [ ]:
np.allclose(adata.X.sum(axis=1), 10_000)

In [ ]:
adata.write_h5ad("/Users/soeren.becker/repos/HumanCytokineDict/data/mouse_cytokine_dict/mouse_cytokine_dict_all_normalized.h5ad")

In [ ]:
pbulks = []
for celltype in ["B_cell"]: #tqdm(adata.obs.celltype.unique()):
    pbulk = decoupler.get_pseudobulk(
        adata[adata.obs.celltype == celltype],
        sample_col="rep",
        groups_col=["celltype", "cytokine"],
        mode="mean",
        min_cells=0,
        min_counts=0,
        min_prop=0,
        min_smpls=0,
        remove_empty=True,
    )
    pbulks.append(pbulk)
    pbulk.write_h5ad(
        f"/Users/soeren.becker/repos/HumanCytokineDict/data/mouse_cytokine_dict/mouse_cytokine_dict_all_normalized_10k_pbulk_{celltype}.h5ad"
    )

In [ ]:
pbulk_all = anndata.concat(pbulks, axis=0)

In [ ]:
pbulk_all.write_h5ad("/Users/soeren.becker/repos/HumanCytokineDict/data/mouse_cytokine_dict/mouse_cytokine_dict_all_normalized_10k_pbulk.h5ad")

### compare to reported lfcs

In [ ]:
mcd = []
xl = pd.ExcelFile("/Users/soeren.becker/repos/HumanCytokineDict/data/mouse_cytokine_dict/41586_2023_6816_MOESM5_ESM.xlsx")
celltypes = xl.sheet_names
for celltype in tqdm(celltypes):
    mcd.append(pd.read_excel("/Users/soeren.becker/repos/HumanCytokineDict/data/mouse_cytokine_dict/41586_2023_6816_MOESM5_ESM.xlsx", sheet_name=celltype))
mcd = pd.concat(mcd, axis=0)
num_degs_per_conditions = mcd.groupby(["Celltype_Str", "Cytokine_Str"]).Gene.nunique()

In [ ]:
order_y

In [ ]:
from matplotlib.colors import LogNorm, Normalize

num_degs_per_conditions_pivot = num_degs_per_conditions.reset_index().pivot(
    index="Cytokine_Str",
    columns="Celltype_Str",
    values="Gene",
).fillna(0)

order_y = num_degs_per_conditions_pivot.mean(axis=1).sort_values(ascending=False).index
order_x = num_degs_per_conditions_pivot.mean(axis=0).sort_values(ascending=False).index

num_degs_per_conditions_pivot = num_degs_per_conditions_pivot.loc[order_y, order_x]

fig, ax = plt.subplots(1,1,figsize=(5, 9))
sns.heatmap(
    num_degs_per_conditions_pivot,
    square=True,
    ax=ax,
    norm=LogNorm(),
)
ax.set_xticks(0.5+np.arange(num_degs_per_conditions_pivot.shape[1]), [l.get_text().replace("_", " ") for l in ax.get_xticklabels()], fontsize=6, family="Arial")
ax.set_yticks(0.5+np.arange(num_degs_per_conditions_pivot.shape[0]), [replace_greek(l) for l in num_degs_per_conditions_pivot.index], fontsize=6, family="Arial")
plt.xlabel("")
plt.ylabel("")
plt.show()

In [ ]:
mean_per_cytokine[mean_per_cytokine>10].index

In [ ]:
from matplotlib.colors import LogNorm, Normalize

num_degs_per_conditions_pivot = num_degs_per_conditions.reset_index().pivot(
    index="Cytokine_Str",
    columns="Celltype_Str",
    values="Gene",
).fillna(0)

mean_per_cytokine = num_degs_per_conditions_pivot.mean(axis=1).sort_values(ascending=False)
order_y = mean_per_cytokine[mean_per_cytokine>4].index

order_x = num_degs_per_conditions_pivot.mean(axis=0).sort_values(ascending=False).index

num_degs_per_conditions_pivot = num_degs_per_conditions_pivot.loc[order_y, order_x]

fig, ax = plt.subplots(1,1,figsize=(5, 6))
ax.set_facecolor("lightgray")
sns.heatmap(
    num_degs_per_conditions_pivot,
    square=True,
    ax=ax,
    norm=LogNorm(),
)
ax.set_xticks(0.5+np.arange(len(order_x)), [l.replace("_", " ") for l in order_x], fontsize=6, family="Arial")
ax.set_yticks(0.5+np.arange(len(order_y)), [replace_greek(l) for l in order_y], fontsize=6, family="Arial")
plt.xlabel("")
plt.ylabel("")
plt.show()

In [ ]:
all_celltypes= ['T_cell_CD4', 'T_cell_CD8', 'B_cell', 'Treg', "ILC", "Monocyte", "NK_cell", "pDC", "cDC1", "cDC2"]

In [ ]:
_ = sc.read_h5ad("/Users/soeren.becker/repos/HumanCytokineDict/data/mouse_cytokine_dict/mouse_cytokine_dict_all_normalized_10k_pbulk.h5ad")

In [ ]:
all_cytokines = _.obs.loc[~_.obs.cytokine.isna()].cytokine.unique()
all_cytokines

In [ ]:
mean_per_species = True
threshold_expression = 1e-5
threshold_min_num_cells = -1
apply_log2 = True

scores = pd.DataFrame()

for celltype in tqdm(all_celltypes):
    
    mcd = pd.read_excel("/Users/soeren.becker/repos/HumanCytokineDict/data/mouse_cytokine_dict/41586_2023_6816_MOESM5_ESM.xlsx", sheet_name=celltype)
    mcd.loc[:, "cytokine"] = mcd.loc[:, "Cytokine_Str"].map(lambda x: cytokine_name_map.get(x,x))
    mcd_celltype = mcd.loc[mcd.Celltype_Str == celltype]
    
    for cytokine in tqdm(all_cytokines, leave=False):

        data = load_data(celltype_combination, cytokine, mouse_to_human_dict=None, load_human=False)

        if data["mouse_cyto"].shape[0] == 0:
            continue
        
        with warnings.catch_warnings():    
            warnings.filterwarnings("ignore", category=FutureWarning)
            warnings.filterwarnings("ignore", category=ImplicitModificationWarning)
            data["mouse_cyto"].obs.loc[:, "donor"] = "mean"
            data["mouse_pbs"].obs.loc[:, "donor"] = "mean"
            
        fcs_mouse = compute_fold_change(
            data["mouse_cyto"], 
            data["mouse_pbs"], 
            threshold_expression=threshold_expression, 
            threshold_min_num_cells=threshold_min_num_cells, 
            apply_log2=apply_log2, 
            pool_cyto_across_donors=True, 
            pool_pbs_across_donors=True
        )
        if fcs_mouse is None:
            continue
        
        mcd_celltype_cytokine = mcd_celltype.loc[mcd_celltype.cytokine == cytokine]
        shared_genes = np.intersect1d(fcs_mouse.var_names, mcd_celltype_cytokine.Gene)
        if len(shared_genes) > 2:
            pcc = pearsonr(
                fcs_mouse[:, shared_genes].X.squeeze(), 
                mcd_celltype_cytokine.set_index("Gene").loc[shared_genes, "Avg_log2FC"]
            )[0]
            scc = spearmanr(
                fcs_mouse[:, shared_genes].X.squeeze(), 
                mcd_celltype_cytokine.set_index("Gene").loc[shared_genes, "Avg_log2FC"]
            )[0]
            row_idx = scores.shape[0]
            scores.loc[row_idx, "celltype"] = celltype
            scores.loc[row_idx, "cytokine"] = cytokine
            scores.loc[row_idx, "num_shared_genes"] = len(shared_genes)
            scores.loc[row_idx, "num_cells"] = fcs_mouse.obs.psbulk_n_cells.values[0]
            scores.loc[row_idx, "num_replicates"] = data["mouse_cyto"].shape[0]
            scores.loc[row_idx, "pearson"] = pcc
            scores.loc[row_idx, "spearman"] = scc

scores.loc[:, "mean_per_species"] = mean_per_species
scores.loc[:, "threshold_expression"] = threshold_expression
scores.loc[:, "threshold_min_num_cells"] = threshold_min_num_cells
scores.loc[:, "apply_log2"] = apply_log2

In [ ]:
sns.heatmap(
    scores.pivot(
        columns="cytokine",
        index="celltype",
        values="pearson"
    ),
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
)
plt.xlabel("")
plt.ylabel("")
plt.show()

In [ ]:
display(scores.groupby("celltype").pearson.mean().sort_values().to_frame())
display(scores.groupby("celltype").spearman.mean().sort_values().to_frame())

In [ ]:
display(scores.groupby("cytokine").pearson.mean().sort_values().to_frame())
display(scores.groupby("cytokine").spearman.mean().sort_values().to_frame())

In [ ]:
sns.kdeplot(
    data=scores,
    x="num_shared_genes",
    y="pearson",
    fill=True
)
sns.scatterplot(
    data=scores,
    x="num_shared_genes",
    y="pearson",
    hue="celltype",
)

# plt.xscale("log")
plt.show()

In [ ]:
sns.scatterplot(
    data=scores,
    x="num_shared_genes",
    y="spearman",
)
plt.xscale("log")
plt.show()

In [ ]:
sns.kdeplot(
    data=scores,
    x="num_shared_genes",
    y="spearman",
)
plt.show()

In [ ]:
sns.histplot(
    data=scores,
    x="num_shared_genes",
    y="spearman",
)
plt.show()

### Response magnitude

In [ ]:
human_to_mouse_dict, mouse_to_human_dict = get_gene_conversion_dict()

In [ ]:
adata = load_mouse_data(cytokine_name_map, from_scratch=False, save=False)

In [ ]:
adata.obs.celltype.unique()

In [ ]:
adata.obs.cytokine.unique().tolist()

In [ ]:
df_magnitudes = pd.DataFrame()

shared_celltype = [
    'T_cell_CD4',
    'T_cell_CD8',
    'B_cell',
    'Treg',
    'ILC',
    'Monocyte',
    'NK_cell',
    'pDC',
    'cDC1',
    'cDC2'
]

for celltype in tqdm(shared_celltype):
    for cytokine in tqdm(adata.obs.cytokine.unique(), leave=False):
        
        adata_pbs, adata_cyto = load_data_mouse(celltype_m=celltype, cytokine=inverse_replace_greek(cytokine), mouse_to_human_dict=mouse_to_human_dict)
        if adata_cyto.shape[0] < 1:
            continue
        adata_pbs_mean = pool_adata(adata_pbs)
        adata_cyto_mean = pool_adata(adata_cyto)
        d = euclidean(adata_pbs_mean.X.squeeze(), adata_cyto_mean.X.squeeze())
        
        row_idx = df_magnitudes.shape[0]
        df_magnitudes.loc[row_idx, "cytokine"] = cytokine
        df_magnitudes.loc[row_idx, "celltype"] = celltype
        df_magnitudes.loc[row_idx, "distance"] = d
        df_magnitudes.loc[row_idx, "num_cells"] = adata_cyto_mean.obs.psbulk_n_cells.values[0]

In [ ]:
winsorization_thresholds = dict(df_magnitudes.groupby("celltype").distance.quantile(0.95))
display(winsorization_thresholds)
df_magnitudes.loc[:, "distance_winsorized"] = df_magnitudes.apply(
    lambda row: 
        winsorization_thresholds[row.celltype] if row.distance > winsorization_thresholds[row.celltype] 
        else row.distance
    , 
    axis=1
)
df_magnitudes.loc[:, "distance_normalized"] = df_magnitudes.apply(lambda row: row["distance_winsorized"] / winsorization_thresholds[row["celltype"]], axis=1)
df_magnitudes

In [ ]:
df_pivot = df_magnitudes.pivot(
    index="cytokine",
    columns="celltype",
    values="distance_normalized",
)

order_y = df_pivot.mean(axis=1).sort_values(ascending=False).index
order_x = df_pivot.mean(axis=0).sort_values(ascending=False).index
df_pivot = df_pivot.loc[order_y, order_x]
fig, ax = plt.subplots(1,1,figsize=(6, 8))
sns.heatmap(
    data=df_pivot,
    cmap="Reds",
    ax=ax,
    square=True,
    cbar=True,
)
plt.xlabel("")
plt.ylabel("")
ax.set_yticks(0.5 + np.arange(df_pivot.shape[0]), df_pivot.index, fontsize=6, family="Arial")
ax.set_xticks(0.5 + np.arange(df_pivot.shape[1]), df_pivot.columns, fontsize=6, family="Arial")
plt.show()

In [ ]:
df_pivot.to_csv("./response_magnitudes_mouse.csv")